In [13]:
# project root to src dir setup
import sys
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(f"Success! Project root added: {parent_dir}")
print("Available folders here:", os.listdir(parent_dir))


Success! Project root added: c:\Users\arsal\OneDrive\Desktop\Study Stuff\Projects\AIML\Data Analysis\solar-performance-intelligence
Available folders here: ['.git', '.gitignore', 'data', 'LICENSE', 'README.md', 'scripts', 'solar-power-generation-and-energy-consumption-data.zip', 'src']


In [21]:
from src.data.loader import DataLoader
from src.data.transformer import DataTransformer
from src.config.settings import RAW_DIR
from src.utils.profiler import profile_dataset

In [18]:
loader = DataLoader(RAW_DIR)
datasets = loader.load_all()

In [25]:
generation = (
        DataTransformer
        .transform_generation(
            datasets["generation"]
        )
        # .pipe(DataTransformer.optimize_dataframe)
    )
    
weather = (
    DataTransformer
    .transform_weather(
        datasets["weather"]
    )
    # .pipe(DataTransformer.optimize_dataframe)
)

irradiance = (
    DataTransformer
    .transform_irradiance(
        datasets["irradiance"]
    )
    # .pipe(DataTransformer.optimize_dataframe)
)

sites = (
    DataTransformer
    .transform_sites(
        datasets["sites"]
    )
    # .pipe(DataTransformer.optimize_dataframe)
)

# Data Profiling
I made a extensive report with shape, nulls and nulls pct, duplicate rows, and memory consumption done by it

In [26]:
for name, df in datasets.items():
    profile_dataset(name=name, df=df)


> GENERATION
Shape:  (2731946, 4)
Memory(Mb):  260.54
Duplicate Rows:  0

Null Report:
                  missing_count  missing_pct
SolarGeneration        1536301    56.234677
CampusKey                    0     0.000000
SiteKey                      0     0.000000
Timestamp                    0     0.000000

> WEATHER
Shape:  (371769, 8)
Memory(Mb):  46.8
Duplicate Rows:  0

Null Report:
                      missing_count  missing_pct
WindSpeed                   162890    43.814842
WindDirection               162890    43.814842
AirTemperature              107113    28.811708
ApparentTemperature         107113    28.811708
RelativeHumidity            107113    28.811708
DewPointTemperature         107113    28.811708
CampusKey                        0     0.000000
Timestamp                        0     0.000000

> IRRADIANCE
Shape:  (102360, 4)
Memory(Mb):  15.31
Duplicate Rows:  0

Null Report:
                missing_count  missing_pct
CloudOpacity               0          0.0
Ghi  

The below was done in order to understand weather there was some due to power outages or some timely shutting off of the devices

In [24]:
import pandas as pd

generation = pd.read_csv(RAW_DIR / "Solar_Energy_Generation.csv")

generation["Timestamp"] = pd.to_datetime(generation["Timestamp"])

nulls_by_hour = generation[
    generation["SolarGeneration"].isna()
]["Timestamp"].dt.hour.value_counts().sort_index()

print(nulls_by_hour)

print("\n\n\n" ,generation.groupby(
    "SiteKey"
)["SolarGeneration"].apply(
    lambda x:
    x.isnull().mean() * 100
).sort_values(ascending=False))

Timestamp
0     114050
1     114447
2     114325
3     114464
4     114438
5     114297
6     103891
7      56296
8      14685
9       8127
10      7294
11      8368
12     13793
13     17241
14     13818
15      8040
16      8342
17     29713
18     53961
19     67187
20    100491
21    113143
22    113042
23    112848
Name: count, dtype: int64



 SiteKey
7     74.543300
5     61.942179
9     60.924873
35    59.736378
29    59.141102
28    59.111230
13    59.083089
4     58.748279
16    58.520075
34    57.996847
21    56.244164
32    56.188134
39    56.105622
14    55.998687
26    55.984250
27    55.955141
17    55.702546
30    55.640545
36    55.511112
42    55.455818
20    55.100230
22    55.097118
37    55.087779
40    55.006848
38    54.916578
31    54.838760
33    54.728257
23    54.603066
24    54.569508
15    54.555521
12    54.502704
19    54.320595
25    54.058307
10    54.048841
3     53.809302
18    53.760329
2     53.649189
41    53.084381
8     52.969720
1     52.630517


# Some quick inference
## Irradiance
This has 0% missing data so it means for now we can trust it

## Sites
has 42 rows of metadata about the sites, with missing columns being Panel, Inverter, Optimizers, Metric, Number of Panels, kWp with somewhere as shown above in the report by profile_dataset() function output
Features like Panel, Inverter, Optimizer, and Metric isnt really of use for 

## Weather
the columns Tempearture, Humidity, Dewpoint being approx 29% missing, and Wind being around 44% missing makes it so we can infer that there is either sensor outages, or a partial collection period only.

## 

In [ ]:
null_pct_by_site = sites_clean.groupby('CampusKey').apply(lambda group: group.isna().mean() * 100)

print(null_pct_by_site[['kWp', 'Number of panels']])

                  kWp  Number of panels
CampusKey                              
1            7.407407          7.407407
2          100.000000        100.000000
3          100.000000        100.000000
4          100.000000        100.000000
5          100.000000        100.000000


# Inference
Technical site metadata is only available for the Bundoora campus. Installed capacity and equipment information are unavailable for all other campuses. Consequently, site-level technical specifications were excluded from the first modeling iteration to maximize dataset coverage

In [39]:
generation.groupby('CampusKey').apply(lambda x: x.isnull().mean() * 100)

,SiteKey,Timestamp,SolarGeneration
CampusKey,,,
1,0.0,0.0,55.866317
2,0.0,0.0,55.346629
3,0.0,0.0,58.617021
4,0.0,0.0,53.084381
5,0.0,0.0,55.455818


# Inference
since we can see that the missing rate for solar generation is ranging between 53 to 58 we can conclude that it is probably a systemic issue

In [40]:
generation["SolarGeneration"].describe()

count    1.195645e+06
mean     6.944463e+00
std      1.120282e+01
min      1.000000e-01
25%      1.128906e+00
50%      3.328125e+00
75%      8.062500e+00
max      9.921880e+01
Name: SolarGeneration, dtype: float64

# inference
snice the mean > median, we know that the dataset is right skewed <br>
as a few high values are pulling the mean upwards as seen with the quartile analysis.<br><br>
25%ile means we see most generation near sunrise/sunset<br>
50%ile gives us the typical geneartion of power<br>
75%ile gives us the top qurater which exceeeds 8.06kWh<br>
now 95%ile and 99%ile shows a huge number like 24.204 and 65.875 kWh respectively, which shows the occasional production spikes like during Sunny Midday periods.<br>


so in summary: <br>
Solar Generation exhibits highly right-skewed distribution. Most Production intervals generate relatively low amounts of energy, while a small number of periods; likely corresponding to optimal irradiance conditions, produce substantially higher outputs.

In [41]:
generation["SolarGeneration"].quantile(
    [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

0.01     0.125000
0.05     0.226562
0.25     1.128906
0.50     3.328125
0.75     8.062500
0.95    24.204688
0.99    65.875000
Name: SolarGeneration, dtype: float64

In [43]:
generation[
    generation["SolarGeneration"].notna()
]["Timestamp"].dt.hour.value_counts().sort_index()

Timestamp
0         39
1         47
2         31
3         28
4         32
5         29
6      10381
7      57940
8      99386
9     105833
10    106612
11    105457
12     99906
13     96398
14     99806
15    105584
16    105329
17     83929
18     59511
19     46285
20     12981
21        38
22        31
23        32
Name: count, dtype: int64

## Generation Data Quality Findings

The SolarGeneration variable contains approximately 56.2% missing values.

Analysis of hourly distributions shows that valid generation records are concentrated between approximately 06:00 and 20:00, corresponding to daylight hours.

Generation values exhibit a strongly right-skewed distribution, with a median production of 3.33 kWh and occasional high-output periods reaching up to 99.2 kWh.

The consistency of missingness across all campuses and sites suggests that missing values are likely related to solar production cycles and data collection practices rather than isolated sensor failures.

## hypothesis
I need to see whether the missing values fall into the systemic issues we think or not as per time including night time.

In [44]:
generation["Hour"] = generation["Timestamp"].dt.hour

hourly_missing = (
    generation
    .groupby("Hour")["SolarGeneration"]
    .apply(lambda x: x.isna().mean() * 100)
)

print(hourly_missing)

Hour
0     99.965816
1     99.958950
2     99.972892
3     99.975544
4     99.972045
5     99.974634
6     90.915535
7     49.280437
8     12.873561
9      7.131450
10     6.403526
11     7.351636
12    12.131153
13    15.171728
14    12.161163
15     7.075970
16     7.338723
17    26.146143
18    47.554463
19    59.210202
20    88.560173
21    99.966425
22    99.972584
23    99.971651
Name: SolarGeneration, dtype: float64


# Inference
As the above shows, it can be seen the majority readings are from 8 - 17, while some decent readings are shown from 7-8 and 17-19
leading to prove that majority missing values are indeed due to nighttime

## conclusion
Missing values in SolarGeneration are strongly associated with non-daylight hours. Missingness exceeds 99% during late-night periods and falls below 10% during peak daylight hours. This suggests that a substantial proportion of missing values correspond to periods when solar production is physically impossible rather than sensor failure.

In [45]:
generation["Hour"] = (
    generation["Timestamp"]
    .dt.hour
)

In [46]:
DAYLIGHT_HOURS = range(7, 20)

In [47]:
daytime = generation[
    generation["Hour"].between(7, 19)
]

daytime["SolarGeneration"].isna().mean() * 100

np.float64(20.750371405715693)

# Problem
we found out majority (~35%) is missing due to nightime, BUT during daytime we are seeing an approx ~21% missing data, which makes us question what happened to the 21% lets see that

In [48]:
daytime = generation[
    generation["Hour"].between(7, 19)
]

daytime.groupby(
    "SiteKey"
)["SolarGeneration"].apply(
    lambda x: x.isna().mean() * 100
).sort_values(
    ascending=False
)

SiteKey
7     53.734575
5     30.168537
9     28.521537
35    26.508385
13    25.760906
28    24.989982
29    24.867709
4     24.291245
16    23.607354
34    23.004011
27    22.466318
14    20.628593
21    20.354873
32    20.274351
26    20.113306
39    19.933890
17    19.558278
30    19.348339
36    19.267823
20    18.850833
22    18.790441
37    18.678285
40    18.675409
42    18.656577
38    18.413712
31    18.333190
33    18.232537
15    18.187502
23    18.085944
24    18.039859
19    17.914525
12    17.848661
25    17.782347
18    16.873301
10    16.731083
2     16.430733
3     16.349243
8     15.321766
41    15.299185
1     14.814901
6     14.526686
11    13.665715
Name: SolarGeneration, dtype: float64

In [49]:
daytime.groupby(
    daytime["Timestamp"].dt.month
)["SolarGeneration"].apply(
    lambda x: x.isna().mean() * 100
)

Timestamp
1     10.210672
2     10.247221
3     16.282824
4     24.783593
5     32.254681
6     36.405498
7     38.714555
8     28.997114
9     21.673160
10    14.243547
11    13.829662
12    12.479644
Name: SolarGeneration, dtype: float64

In [50]:
daytime.groupby(
    daytime["Timestamp"].dt.year
)["SolarGeneration"].apply(
    lambda x: x.isna().mean() * 100
)

Timestamp
2020    23.712447
2021    20.942786
2022    14.655256
Name: SolarGeneration, dtype: float64

In [53]:
for i in range(42):
    print("="*60)
    print(f"site = {i+1}")
    print("="*60)
    sitedata = daytime[daytime["SiteKey"] == (i+1)]
    nulld = sitedata[
            sitedata["SolarGeneration"].isna()
        ].head(10)
    print(nulld)
    print("="*60)
    print("\n\n")

site = 1
      CampusKey  SiteKey           Timestamp  SolarGeneration  Hour
54            2        1 2020-01-01 13:45:00              NaN    13
55            2        1 2020-01-01 14:00:00              NaN    14
56            2        1 2020-01-01 14:15:00              NaN    14
57            2        1 2020-01-01 14:30:00              NaN    14
411           2        1 2020-01-05 07:00:00              NaN     7
412           2        1 2020-01-05 07:15:00              NaN     7
1006          2        1 2020-01-11 11:45:00              NaN    11
1007          2        1 2020-01-11 12:00:00              NaN    12
1008          2        1 2020-01-11 12:15:00              NaN    12
1009          2        1 2020-01-11 12:30:00              NaN    12



site = 2
       CampusKey  SiteKey           Timestamp  SolarGeneration  Hour
79372          2        2 2020-01-01 13:30:00              NaN    13
79373          2        2 2020-01-01 13:45:00              NaN    13
79374          2        

# Inference
1. what have we found till now from the consecutive block analysis that we can see some consecutive chunks missing out not just random points which gives the following points as inference.
- It is not MCAR
- likely sensor outages
- possible communication failures
- possible maintainance windows
2. The Month wise analysis shows us that the missing values dramatically increases during winter, july-aug is winter in australia btw
- summer shows ~10% missing values while winter shows ~25% missing values
Now we draw 3 Hypothesis for this:
A. Lower Sunlgiht -> system not recording low outputs
B. Weather conditions causing outages
C. Solar system were commissioned gradually
3. As year went by we can see a gradual increase in data quality
4. Site 7 is extremely suspicious with around ~54% of its recordings missing during daytime, and in comparison to other sites its performing extremely bad.
5. Sites with missing metadata like inverter kwp or panel are an issue too, leading to a probable data incompleteness.

## Strategy:
Since we want Solar Geneartion using weather, irradiance, and sites its better to not focus on reconstruction of data which we cannot reconstruct.
Cleaning Strategy:
- During Nightime: fill data with 0 as its a correct reading since no sunlight means no generation
- During Daytime: no need for complete interpolation, as it would introduce fake data in the reading, better go for dropping daytime NaNs 
## Conclusion:
Missing generation values exhibit strong temporal structure rather than random behavior. Approximately 56% of observations are missing overall, largely due to nighttime periods where solar production is physically zero. Remaining daytime missingness (~21%) occurs in contiguous blocks and varies substantially by site, month, and year, suggesting a combination of operational outages, maintenance events, and incomplete historical records. For modeling purposes, nighttime missing values will be treated as zero generation, while daytime missing values will be excluded from supervised training to avoid introducing synthetic targets.

# Some Weather related Analysis

In [54]:
weather.isna().mean() * 100

CampusKey               0.000000
Timestamp               0.000000
ApparentTemperature    28.811708
AirTemperature         28.811708
DewPointTemperature    28.811708
RelativeHumidity       28.811708
WindSpeed              43.814842
WindDirection          43.814842
dtype: float64

In [55]:
weather.isna().groupby(
    weather["CampusKey"]
).mean() * 100

,CampusKey,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
CampusKey,,,,,,,,
1,0.0,0.0,18.261599,18.261599,18.261599,18.261599,36.646630,36.646630
2,0.0,0.0,15.753484,15.753484,15.753484,15.753484,32.143871,32.143871
3,0.0,0.0,10.790328,10.790328,10.790328,10.790328,22.066974,22.066974
4,0.0,0.0,43.721688,43.721688,43.721688,43.721688,55.126702,55.126702
5,0.0,0.0,74.193413,74.193413,74.193413,74.193413,93.536823,93.536823


In [56]:
weather.isna().groupby(
    weather["Timestamp"].dt.month
).mean() * 100

,CampusKey,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
Timestamp,,,,,,,,
1,0.0,0.0,25.688844,25.688844,25.688844,25.688844,43.937212,43.937212
2,0.0,0.0,31.649832,31.649832,31.649832,31.649832,49.494949,49.494949
3,0.0,0.0,31.972446,31.972446,31.972446,31.972446,50.000000,50.000000
4,0.0,0.0,30.209885,30.209885,30.209885,30.209885,45.402844,45.402844
5,0.0,0.0,22.222222,22.222222,22.222222,22.222222,22.222222,22.222222
6,0.0,0.0,22.388117,22.388117,22.388117,22.388117,22.550154,22.550154
7,0.0,0.0,29.330944,29.330944,29.330944,29.330944,35.084379,35.084379
8,0.0,0.0,32.907706,32.907706,32.907706,32.907706,44.444444,44.444444
9,0.0,0.0,32.935957,32.935957,32.935957,32.935957,44.444444,44.444444


In [58]:
weather[weather["AirTemperature"].isna()]

,CampusKey,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
26600,1,2020-10-04 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
26601,1,2020-10-04 02:15:00,NaN,NaN,NaN,NaN,NaN,NaN
26602,1,2020-10-04 02:30:00,NaN,NaN,NaN,NaN,NaN,NaN
26603,1,2020-10-04 02:45:00,NaN,NaN,NaN,NaN,NaN,NaN
32217,1,2020-12-01 14:15:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
371759,5,2022-04-23 19:45:00,NaN,NaN,NaN,NaN,NaN,NaN
371761,5,2022-04-23 20:15:00,NaN,NaN,NaN,NaN,NaN,NaN
371763,5,2022-04-23 20:45:00,NaN,NaN,NaN,NaN,NaN,NaN
371765,5,2022-04-23 21:15:00,NaN,NaN,NaN,NaN,NaN,NaN


# Inference
What i can see is that during the anaalysis of weather data <br>
when i did the analysis with campus in mind, saying missing values per campus, we can see that the missing values pct differ from each other so this means it is NOT MCAR, but either MAR or MNAR. Insane is the part campus 5 and 3 got 74% and 10% missing rate, cuz the difference is high <br>
Other thing that can be seen is that in the chunk test we did (manual analysis) we can find that the data is always missing in chunks and not isolated, so the missigness is definitely systemic. This can lead to 3 possibilities, API Outage, Sensor Outage, or Data Collection Failure <br>
By Month Anatlysis it can be shown that during Winter (June-Aug) can be seen between 30-50%, while in summer it can be seen between 20-25%, again the data is systemic and follows weather patterns, so it can be confidently said it is NOT MCAR.<br>

In [59]:
weather["missing"] = (
    weather["AirTemperature"]
    .isna()
)

weather["group"] = (
    weather["missing"] != weather["missing"].shift()
).cumsum()

gaps = (
    weather[weather["missing"]]
    .groupby("group")
    .size()
)

In [60]:
gaps.describe()

count    53564.000000
mean         1.999720
std        161.667004
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max      26457.000000
dtype: float64

# Report Conclusion:
Missing-value treatment was determined empirically through gap-length analysis. Over 75% of weather-data missing segments consisted of a single timestamp and were reconstructed via short-range interpolation. Extremely long gaps (up to ~275 days) were preserved as missing values to avoid introducing unrealistic synthetic observations. Solar generation values were handled using domain knowledge: nighttime missing values were assigned zero generation, while daytime missing values were retained and excluded from supervised learning due to their likely association with operational outages rather than true production levels.